# 4. Geospatial processing

### [read] Summarise data availability of addresses among the fixed_table
- So we know what we're working with in terms of different types

In [6]:
# ### [read] Summarise data availability of addresses among the fixed_table
# - So we know what we're working with in terms of different types
# This should be lat/long, full address, address decomposed across the different fields, just postcode, the nothing
# And if its primary trading address or registered office address
import ibis

address_hierachy = [
    {'primary_trading_address_latitude', 'primary_trading_address_longitude'},
    {'primary_trading_address'},
    {'ro_latitude', 'ro_longitude'},
    {'ro_address'},
    {'ro_address_line_1', 'ro_full_postcode'},
    {'ro_full_postcode'},
    {'ro_postcode'}
]

# Dynamically builds an ibis.cases() expression from a list of column sets.
def build_location_source_cases(table: ibis.expr.types.Table, hierarchy: list[set]):
    cases_list = []
    
    for i, field_set in enumerate(hierarchy, start=1):
        # 1. Build the logical condition: EVERY column in the set must be not-null
        condition = None
        for col_name in field_set:
            is_not_null = table[col_name].notnull()
            condition = is_not_null if condition is None else condition & is_not_null
        
        # 2. Store as a (condition, result_value) tuple
        cases_list.append((condition, i))
        
    # 3. Unpack the list of tuples into ibis.cases. 
    # Fallback MUST be an integer to match the column type.
    fallback_lvl = len(hierarchy) + 1
    return ibis.cases(*cases_list, else_=fallback_lvl)

### [write] Add the ONS postcode directory to the database

In [ ]:
# Add input/NSPL_MAY_2026_UK.csv as a table to the database
# Keep only columns pcds, lat, and long
import pandas as pd
from numpy.random import rand
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="descriptives")
table_name = "ref_ons_postcode"
file_name = "NSPL_MAY_2026_UK.csv"
keep_cols = { "pcds", "lat", "long", "ttwa15cd" }

# # Skip 99.9% of the rows
# csv_file = f"/mnt/c/Users/lazym/OneDrive - University College London/Diss 2/Data/Postcode/{file_name}"
# df_raw = pd.read_csv(csv_file, skiprows=lambda x: x > 0 and rand() > 0.001)
# df_raw_short = df_raw.sample(500)
# df_raw_short.to_csv(f"{dirs.input_dir}/short_{file_name}", index=False)

csv_file = f"{dirs.input_dir}/{file_name}"
df_raw = pd.read_csv(csv_file, usecols=list(keep_cols))
# df_raw.to_csv(f"{dirs.input_dir}/skinny_{file_name}")
if len(set(df_raw.columns).intersection(keep_cols)) != len(keep_cols):
    for col in keep_cols:
        if col not in set(df_raw.columns):            
            raise ValueError(f"Expected {col} in columns: missing")

codes_length = len(df_raw)
display(df_raw.sample(20))

❌ ROOT_DATA_DIR path does not exist or is not a directory: /mnt/h/Other computers/My computer/fame_clean


In [13]:
import ibis

# Initialize connection
con = ibis.duckdb.connect(str(dirs.db_path))

con.create_table(table_name, df_raw, overwrite=table_name in con.list_tables())
table_ref = con.table(table_name)
rows_count = table_ref.count().execute()
print(f"{rows_count:,} rows in new table {table_name}")
print("Sample:")
display(table_ref.sample(10 / rows_count).execute())

2,726,477 rows in new table ref_ons_postcode
Sample:


,pcds,ttwa15cd,lat,long
0,G41 4WA,S22000065,55.834600,-4.285013
1,G74 3HJ,S22000065,55.773903,-4.149340
2,KT15 9AE,E30000266,51.371324,-0.489221
3,NE5 3PH,E30000245,54.991128,-1.670654
4,NN15 6EL,E30000224,52.389562,-0.719436
5,SW16 4QL,E30000234,51.408579,-0.136814
6,TW18 1TY,E30000266,51.434063,-0.530096
7,W1U 3QB,E30000234,51.517472,-0.154398


### [write] Create new column for address geocoding
- The third decimal place is worth up to 110 m: it can identify a large agricultural field or institutional campus.
- The fourth decimal place is worth up to 11 m: it can identify a parcel of land. It is comparable to the typical accuracy of an uncorrected GPS unit with no interference.
- The fifth decimal place is worth up to 1.1 m: it distinguish trees from each other. Accuracy to this level with commercial GPS units can only be achieved with differential correction.

In [31]:
import ibis
from ibis import _
from utils.f_0_dirs import get_data_dirs
from f_3_spatial import convert_dms_to_decimal

old_table_name = "fame_fixed_filtered"
new_table_name = "working_fixed"

# Initialize connection
dirs = get_data_dirs()
con = ibis.duckdb.connect(str(dirs.db_path))
con.raw_sql("INSTALL spatial; LOAD spatial;")

# Reference the existing tables
fame_fixed = con.table(old_table_name)

# 1. Parse availability into an indicator and extract the best available text address
sep: ibis.StringScalar = ibis.literal(", ", type="string")
working_raw = fame_fixed.mutate(
    address_raw_lvl = build_location_source_cases(fame_fixed, address_hierachy),
    address_raw = ibis.coalesce(
        fame_fixed.primary_trading_address,
        fame_fixed.ro_address,
        # Fallback: concatenate the separate lines and postcode if above are null
        sep.join(
            ibis.array([
                fame_fixed.ro_address_line_1, 
                fame_fixed.ro_address_line_2, 
                fame_fixed.ro_full_postcode
            ]).filter(lambda x: x.notnull())
        ),
        fame_fixed.ro_full_postcode,
        fame_fixed.ro_postcode
    )
)

# 2. Extract and clean postcodes for Levels 2, 4, 5, 6, and 7
# Regex matches standard UK postcode formats. We strip whitespace and uppercase for perfect joins.
# 2. Extract and clean postcodes for Levels 2, 4, 5, 6, and 7
# Regex matches standard UK postcode formats. 
uk_postcode_regex = r"([A-Za-z]{1,2}\d[A-Za-z\d]?\s?\d[A-Za-z]{2})"

working_pc = working_raw.mutate(
    extracted_postcode = ibis.cases(
        (working_raw.address_raw_lvl.isin([1, 2]), working_raw.address_raw.re_extract(uk_postcode_regex, 1)),
        (working_raw.address_raw_lvl.isin([3, 4]), working_raw.ro_address.re_extract(uk_postcode_regex, 1)),
        (working_raw.address_raw_lvl.isin([5, 6]), working_raw.ro_full_postcode),
        (working_raw.address_raw_lvl == 7, working_raw.ro_postcode),
        else_=ibis.literal(None, type="string")
    )
    .cast("string")
    .upper()
    .re_replace(r"\s+", "") # Step 1: Strip all existing spaces (e.g. "EC1Y2AL" or "EC1Y  2AL" -> "EC1Y2AL")
    .re_replace(r"(.+)(\d[A-Z]{2})$", r"\1 \2") # Step 2: Insert exactly one space before the 3-character inward code
)

# Clean the ONS directory postcodes using the same logic for the join
ons_table_name = "ref_ons_postcode" # Assume this exists with 'postcode', 'lat', 'long'
ons_lookup = con.table(ons_table_name) 

# 3. Join firm data with the ONS lookup
working_joined = working_pc.left_join(
    ons_lookup,
    working_pc.extracted_postcode == ons_lookup.pcds
)

# 4. Extract standardized spatial coordinates based on ALL 7 levels of the hierarchy
working_fixed = (
    working_joined
    .mutate(
        address_case = ibis.cases(
            (working_joined.address_raw_lvl <= 2, ibis.literal('pta')),
            (working_joined.address_raw_lvl <= 7, ibis.literal('ro')),
            else_=ibis.literal(None, type="string")
        ),
        address_lvl = ibis.cases(
            (working_joined.address_raw_lvl <= 2, working_joined.address_raw_lvl),
            (working_joined.address_raw_lvl <= 7, working_joined.address_raw_lvl - 2),
            else_=ibis.literal(None, type="int64")
        ),
        lat_dec = ibis.cases(
            (working_joined.address_raw_lvl == 1, convert_dms_to_decimal(working_joined.primary_trading_address_latitude)),
            (working_joined.address_raw_lvl == 3, convert_dms_to_decimal(working_joined.ro_latitude)),
            else_=working_joined.lat # ibis.literal(None, type='float64') # # Automatically covers levels 2, 4, 5, 6, and 7
        ),
        lon_dec = ibis.cases(
            (working_joined.address_raw_lvl == 1, convert_dms_to_decimal(working_joined.primary_trading_address_longitude)),
            (working_joined.address_raw_lvl == 3, convert_dms_to_decimal(working_joined.ro_longitude)),
            else_=working_joined.long # ibis.literal(None, type='float64') # # Automatically covers levels 2, 4, 5, 6, and 7
        ),
        ttwa = working_joined.ttwa15cd,
        pc4 = working_joined.extracted_postcode.split(" ")[0],
    )
    .mutate(
        lat_lon5 = ibis._.lat_dec.round(5).cast("string") + "," + ibis._.lon_dec.round(5).cast("string")
    )
)

working_fixed_loc = (
    working_fixed
    .select('registered_number', 'address_raw', 'extracted_postcode', 'address_case', 'address_lvl', 'lat_dec', 'lon_dec', 'ttwa', 'pc4', 'lat_lon5')
    # .drop("extracted_postcode", "postcode", "lat", "long") # Drop join artifacts
    .mutate(
        # We need to split by " ", not just take the first 4 characters, because some postcodes have a space in the middle (e.g. "EC1A 1BB" -> "EC1A")
        lat_lon4 = working_fixed.lat_dec.round(4).cast("string") + "," + working_fixed.lon_dec.round(4).cast("string"),
        lat_lon3 = working_fixed.lat_dec.round(3).cast("string") + "," + working_fixed.lon_dec.round(3).cast("string")
    )
)

row_count = working_fixed_loc.count().execute()
display(working_fixed_loc.sample(10 / row_count).execute())

# Filter for those which extracted_postcode is NaN
# Count and display those rows (max 50)
nan_rows = working_fixed_loc.filter(working_fixed_loc.extracted_postcode.isnull())
nan_count = nan_rows.count().execute()
print(f"Number of rows with NaN extracted_postcode: {nan_count}={nan_count/row_count*100:.1f}%")
if nan_count > 0:
    display(nan_rows.sample(50 / nan_count).execute())
    for row in nan_rows.limit(50).execute().to_dict(orient='records'):
        print(row)

# Summary stats
# Get unique counts of ttwa, first half of extracted_postcode, full extracted_postcode, and latitude and longitude (combined and rounded to 5 decimal places)
stats_loc = working_fixed_loc.aggregate(
    ttwa_count = _.ttwa.nunique(),
    postcode_first_half_count = _.pc4.nunique(),
    postcode_full_count = _.extracted_postcode.nunique(),
    lat_lon3_count = _.lat_lon3.nunique(),
    lat_lon4_count = _.lat_lon4.nunique(),
    lat_lon5_count = _.lat_lon5.nunique(),
    firms = _.registered_number.nunique()
)
display(stats_loc.execute())

❌ ROOT_DATA_DIR path does not exist or is not a directory: /mnt/h/Other computers/My computer/fame_clean


,registered_number,address_raw,extracted_postcode,address_case,address_lvl,lat_dec,lon_dec,ttwa,pc4,lat_lon5,lat_lon4,lat_lon3
0,03840530,"c/o Baker Tilly, 6th Floor 25 Farringdon Stree...",EC4A 4AB,ro,1,51.515556,-0.104778,E30000234,EC4A,"51.51556,-0.10478","51.5156,-0.1048","51.516,-0.105"
1,13322121,"4th Floor One New Change, London, London, EC4M...",EC4M 9AF,pta,2,51.513624,-0.095953,E30000234,EC4M,"51.51362,-0.09595","51.5136,-0.096","51.514,-0.096"
2,SC236246,"5 South Gyle Crescent Lane, Edinburgh, Midloth...",EH12 9EG,ro,1,55.932056,-3.303444,S22000059,EH12,"55.93206,-3.30344","55.9321,-3.3034","55.932,-3.303"
3,SC097754,"c/o Grainger Corporate Rescue & Recovery, 65 B...",G2 2BX,ro,1,55.863861,-4.256667,S22000065,G2,"55.86386,-4.25667","55.8639,-4.2567","55.864,-4.257"
4,03164559,"Zion Community Resource Centre, 339 Stretford ...",M15 4ZY,ro,1,53.465250,-2.260139,E30000239,M15,"53.46525,-2.26014","53.4653,-2.2601","53.465,-2.26"
5,10930310,"Randalls Farm, Scottlethorpe Road, Edenham, Bo...",PE10 0LN,pta,1,52.775750,-0.441722,E30000054,PE10,"52.77575,-0.44172","52.7758,-0.4417","52.776,-0.442"
6,08492481,"10 Little Portland Street, London, London, W1W...",W1W 7JG,pta,2,51.517195,-0.142028,E30000234,W1W,"51.5172,-0.14203","51.5172,-0.142","51.517,-0.142"
7,05086063,"High Holborn House, 52-54 High Holborn, London...",WC1V 6RL,pta,1,51.518167,-0.114778,E30000234,WC1V,"51.51817,-0.11478","51.5182,-0.1148","51.518,-0.115"
8,10247081,"7 Savoy Court, London, London, WC2R 0EX",WC2R 0EX,pta,2,51.510528,-0.120870,E30000234,WC2R,"51.51053,-0.12087","51.5105,-0.1209","51.511,-0.121"


Number of rows with NaN extracted_postcode: 0=0.0%


,ttwa_count,postcode_first_half_count,postcode_full_count,lat_lon3_count,lat_lon4_count,lat_lon5_count,firms
0,228,2705,49963,48540,53193,53246,148442


In [38]:
working_fixed_skinny = (
    working_fixed    
    .mutate(is_public = working_fixed.ticker_symbol.notnull())
    .select(
        # fame_fixed variables which we want to keep for the regression
        'registered_number', 'company_name', 'is_public', 'industry_codes', 'file_codes',
        'primary_uk_sic_2007_code', 'primary_uk_sic_2007_description',
    
        # new location variables
        'lat_dec', 'lon_dec', 'address_lvl', 'address_case', 'extracted_postcode', 'ttwa', 'pc4', 'lat_lon5'
    )
    .rename(
        pc8='extracted_postcode',
        sic6='primary_uk_sic_2007_code',
        sic6_desc='primary_uk_sic_2007_description'
    )
)

print(f"✅ Inserting columns into new '{new_table_name}' table.")
con.create_table(new_table_name, working_fixed_skinny, overwrite=True)

# Verify the final materialized table
final_table = con.table(new_table_name)
row_count = final_table.count().execute()
col_count = len(final_table.columns)

print(f"✅ Materialized '{new_table_name}' table.")
print(f"📊 Number of rows: {row_count:,}")
print(f"📊 Number of columns: {col_count}")
print(f"\nSample of {new_table_name}:")
display(final_table.sample(30 / row_count).execute())

✅ Inserting columns into new 'working_fixed' table.
✅ Materialized 'working_fixed' table.
📊 Number of rows: 152,379
📊 Number of columns: 15

Sample of working_fixed:


,registered_number,company_name,is_public,industry_codes,file_codes,sic6,sic6_desc,lat_dec,lon_dec,address_lvl,address_case,pc8,ttwa,pc4,lat_lon5
0,02268089,RYBROOK CARS LIMITED,False,45,14_37,45111,Sale of new cars and light motor vehicles,52.596556,-1.187250,1,ro,LE19 1ST,E30000230,LE19,"52.59656,-1.18725"
1,01548338,ACTION HOUSING AND SUPPORT LIMITED,False,96,17_32,96090,Other personal service activities n.e.c.,53.421306,-1.372583,1,pta,S60 1DX,E30000261,S60,"53.42131,-1.37258"
2,04831222,BLACKROW HOLDINGS LIMITED,False,64,12_47 2,64209,Activities of other holding companies (not inc...,53.574956,-0.121526,2,pta,DN31 2TP,E30000211,DN31,"53.57496,-0.12153"
3,05664251,MOBILE EDUCATION PARTNERSHIPS,False,85,17_31 2,85590,Other education n.e.c.,54.724305,-1.564940,2,pta,DH6 5LX,E30000203,DH6,"54.72431,-1.56494"
4,00670006,GSU (BOREHAMWOOD) LIMITED,False,27,22_33,27400,Manufacture of electric lighting equipment,51.279389,-1.107083,1,ro,RG24 9JP,E30000164,RG24,"51.27939,-1.10708"
5,07852122,DENEFIELD SCHOOL,False,85,17_31,85310,General secondary education,51.471611,-1.054722,1,pta,RG31 6XY,E30000256,RG31,"51.47161,-1.05472"
6,07918469,WELL DUNN LIMITED,False,65,16_34,65120,Non-life insurance,53.477778,-2.236306,1,pta,M15 4JJ,E30000239,M15,"53.47778,-2.23631"
7,00182382,RENOLD POWER TRANSMISSION LIMITED,False,"28,25","22_34,22_22",25930,"Manufacture of wire products, chain and springs",53.365500,-2.241806,1,pta,M22 5XB,E30000239,M22,"53.3655,-2.24181"
8,06484132,ADELSTONE DENTAL CARE LIMITED,False,86,17_50 2,86230,Dental practice activities,53.543777,-2.369123,2,pta,M26 1GG,E30000239,M26,"53.54378,-2.36912"
9,01370453,AXGRO FOODS LIMITED,False,10,23_18,10390,Other processing and preserving of fruit and v...,52.483806,-1.895931,2,ro,B4 6AT,E30000169,B4,"52.48381,-1.89593"


# 2. Geospatial processing
**Groups (donut approach)**

1. Same building: address is equal, long/latitude is identical
2. 8-digit postcode is identical
3. 4-digit postcode is identical
4. NUTS-2 region

  5-8. Add 2-digit SIC codes

  9-12. Add 6-digit SIC codes

- Exclude the previous donut regions from each successive one
- Don’t spend too much time on this as this model is poorly identified anyway. Just do basic proof of concept / correlation and discuss the issues.
- **Separate regressions (baseline):** for each geographic grouping and SIC grouping one-by-one (12 models). Establishes baseline gross effect → expect to see weaker effects with each outside group.
- **Combined model:** final, preferred specification is combined model with all mutually exclusive rings (I can drop outer rings but not inner). Same with industry effect, can control. Including industries could distinguish knowledge spillovers (same supply chain, co-located) to localised same product market competition (same SIC code). Roughly correspond to Van Reenen, Bloom, Schankerman (2021).

**Data group assignment**

- roughly $O(nt\log [nt])=O(9m)$
1. Partition: by group string (as laid out above)
2. Aggregation: calculate average of remaining rows = sum -0 count
3. Assign to all filtered rows
4. Transformation: leave-one-out.

In [ ]:
import ibis
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="descriptives")
table_name_fixed = "working_fixed"
table_name_yearly = "working_yearly"
table_name_peers = "working_yearly_with_peers"
column_peer = "gva1_per_worker"       # "tfp"
column_out_name = "tfp"

con = ibis.duckdb.connect(str(dirs.db_path))
con.raw_sql("INSTALL spatial; LOAD spatial;")

table_fixed = con.table(table_name_fixed)
table_yearly = con.table(table_name_yearly)
table_joined = (
    table_yearly
    # Filter positive values on column_peer
    .filter(table_yearly[column_peer] > 0)
    .left_join(
        table_fixed,
        "registered_number"
    )
)
columns_yearly_str = ", ".join(table_yearly.columns)

table_name_view = "spatial_panel_view"
con.create_view(table_name_view, table_joined, overwrite=True)

# ==========================================
# 2. PARTITION, AGGREGATE, BROADCAST, TRANSFORM
# ==========================================
# We use 3 concentric rings. 
# Ring 1: extracted_postcode (Innermost)
# Ring 2: pc4 (Middle Donut)
# Ring 3: ttwa (Outer Donut)
window_query_body = f"""
WITH GroupAggregates AS (
    SELECT 
        {columns_yearly_str},
        {column_peer},
        
        -- 1. Full Postcode Level (Innermost Ring)
        SUM({column_peer}) OVER (PARTITION BY pc8, year) AS sum_pc8,
        COUNT({column_peer}) OVER (PARTITION BY pc8, year) AS count_pc8,
        
        -- 2. 4-Digit Postcode Level (Middle Ring)
        SUM({column_peer}) OVER (PARTITION BY pc4, year) AS sum_pc4,
        COUNT({column_peer}) OVER (PARTITION BY pc4, year) AS count_pc4,
        
        -- 3. TTWA Level (Outer Ring)
        SUM({column_peer}) OVER (PARTITION BY ttwa, year) AS sum_ttwa,
        COUNT({column_peer}) OVER (PARTITION BY ttwa, year) AS count_ttwa
    FROM 
        {table_name_view}
)
SELECT 
    {columns_yearly_str},
    
    -- Ring 1: Peers in exact same postcode (Leave-One-Out)
    (sum_pc8 - {column_peer}) / NULLIF(count_pc8 - 1, 0) AS peer_{column_out_name}_pc8,
    
    -- Ring 2: Peers in same 4-digit PC (Donut)
    (sum_pc4 - sum_pc8) / NULLIF(count_pc4 - count_pc8, 0) AS peer_{column_out_name}_pc4_donut,
    
    -- Ring 3: Peers in TTWA (Donut)
    (sum_ttwa - sum_pc4) / NULLIF(count_ttwa - count_pc4, 0) AS peer_{column_out_name}_ttwa_donut
    
FROM 
    GroupAggregates
"""

# 2. Write new table directly in the raw sql
create_table_ddl = (
    f"CREATE OR REPLACE TABLE {table_name_peers} AS ({window_query_body});"
)

# 3. Execute directly on DuckDB connection
con.raw_sql(create_table_ddl)

# 4. Bind the newly materialized physical table back to Ibis
table_peers = con.table(table_name_peers)
# View the schema to verify the 3 new mutually exclusive spatial donut columns
print(table_peers.schema())
# Print the head of the resulting table to verify the calculations
display(table_peers.sample(0.0001).execute())

ibis.Schema {
  registered_number    string
  year                 int64
  employees            int64
  average_wage         float64
  gva1                 float64
  gva2                 float64
  gva1_per_worker      float64
  gva2_per_worker      float64
  peer_tfp_pc8         float64
  peer_tfp_pc4_donut   float64
  peer_tfp_ttwa_donut  float64
}


,registered_number,year,employees,average_wage,gva1,gva2,gva1_per_worker,gva2_per_worker,peer_tfp_pc8,peer_tfp_pc4_donut,peer_tfp_ttwa_donut
0,02447513,2024,75,23.691560,2678.006000,NaN,35.706747,NaN,NaN,NaN,46.063466
1,00062904,2019,1033,45.456298,60532.376381,52200.102714,58.598622,50.532529,72.820711,81.986953,75.489895
2,03866594,2011,78,37.495155,5374.939850,5560.829955,68.909485,71.292692,103.958778,116.052706,97.763723
3,03544952,2016,120,33.981609,4361.751508,4531.478849,36.347929,37.762324,NaN,37.285313,70.330219
4,09570706,2019,24,38.122708,1930.733267,NaN,80.447219,NaN,217.056675,319.350643,324.644482
...,...,...,...,...,...,...,...,...,...,...,...
103,01427132,2020,1038,2.378749,3034.978235,3013.589456,2.923871,2.903265,NaN,44.381367,166.662751
104,01099088,2016,237,42.479575,21000.400639,NaN,88.609285,NaN,NaN,44.571456,94.239296
105,09332199,2016,37,104.698314,1835.094551,NaN,49.597150,NaN,116.397751,197.836833,208.152815
106,02713318,2016,286,59.525729,15369.024663,NaN,53.737848,NaN,1238.750080,99.262933,209.413712


In [42]:
con.disconnect()